In [ ]:
# ============================================================
# Rot-MNIST Angle Regression – Robustness Benchmark (Colab one-cell)
# ============================================================

!pip -q install torch torchvision tqdm

import os, math, random, time, copy
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import functional as TF

try:
    from torchvision.transforms import InterpolationMode
    BILINEAR = InterpolationMode.BILINEAR
except Exception:
    BILINEAR = 2

# =========================
# Config
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"

modes = [
    "proposed",
    "standard","robust_huber","robust_tukey","trim",
    "dro","awp","rat"
]

N_TRIALS   = 5
EPOCHS     = 5
BATCH_SIZE = 128
LR         = 1e-3
WEIGHT_DECAY = 1e-4
ANGLE_MAX  = 90.0
IMG_SIZE   = 32
OUTLIER_RATIO_TRAIN = 0.05   # Train only
HOVR_M     = 10
OUTDIR     = "./results_adv5"
os.makedirs(OUTDIR, exist_ok=True)

# =========================
# Utils
# =========================
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def to_radians(x): return x * math.pi / 180.0

def corrupt_image(x, p=0.05):
    if np.random.rand() > p:
        return x
    r = np.random.choice(["noise","black","white"])
    if r == "noise":
        x = x + 0.5 * torch.randn_like(x)
    else:
        H,W = x.shape[-2:]
        s = int(np.random.randint(6, min(12, min(H, W)) + 1))
        i = int(np.random.randint(0, max(1, H - s + 1)))
        j = int(np.random.randint(0, max(1, W - s + 1)))
        c = 0.0 if r=="black" else 1.0
        x[..., i:i+s, j:j+s] = c
    return torch.clamp(x, 0, 1)

# =========================
# Dataset
# =========================
class RotAngleMNIST(Dataset):
    def __init__(self, root="./data", train=True, angle_max_deg=90.0, size=32, seed=42, outlier_ratio=0.05):
        base = datasets.MNIST(root=root, train=train, download=True,
                              transform=None)  # 自前で Tensor/resize する
        # raw uint8 -> float tensor [0,1]
        self.images = base.data.unsqueeze(1).float() / 255.0
        self.train = train
        self.size = size
        self.outlier_ratio = outlier_ratio if train else 0.0
        rng = np.random.default_rng(seed if train else seed+1)
        self.angles_deg = rng.uniform(-angle_max_deg, angle_max_deg, len(self.images)).astype(np.float32)

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        x = self.images[idx]                      # (1, 28, 28)
        x = TF.resize(x, [self.size, self.size], interpolation=BILINEAR)
        theta_deg = float(self.angles_deg[idx])
        ## rotate
        x_rot = TF.rotate(x, angle=theta_deg, interpolation=BILINEAR, fill=0.0)
        if self.train and self.outlier_ratio>0:
            x_rot = corrupt_image(x_rot, p=self.outlier_ratio)
        theta = torch.tensor(to_radians(theta_deg), dtype=torch.float32)
        return x_rot, theta

# =========================
# Model
# =========================
class SmallCNN(nn.Module):
    def __init__(self, act="sigmoid"):
        super().__init__()
        self.act = act
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1); self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1); self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1); self.bn3 = nn.BatchNorm2d(128)
        self.gap = nn.AdaptiveAvgPool2d(1); self.fc = nn.Linear(128,1)
    def _act(self,z): return torch.sigmoid(z) if self.act=="sigmoid" else F.relu(z, inplace=True)
    def forward(self,x):
        x = self._act(self.bn1(self.conv1(x))); x = F.max_pool2d(x,2)
        x = self._act(self.bn2(self.conv2(x))); x = F.max_pool2d(x,2)
        x = self._act(self.bn3(self.conv3(x))); x = self.gap(x).flatten(1)
        return self.fc(x).squeeze(1)

# =========================
# Loss
# =========================
def mse_per_sample(p,y): return (p-y)**2
def huber_per_sample(p,y,delta=1.0):
    e=(p-y).abs(); return torch.where(e<delta, 0.5*e**2, delta*(e-0.5*delta))
def tukey_per_sample(p,y,c=4.685):
    r=p-y; a=(r/c).abs(); val=torch.empty_like(r)
    mask=(a<=1); val[mask]=(c**2/6)*(1-(1-a[mask]**2)**3); val[~mask]=(c**2)/6; return val
def trimmed_mean(losses,h=0.9):
    k=max(1,int(h*len(losses))); vals,_=torch.topk(losses,k,largest=False); return vals.mean()
def cvar_mean(losses,alpha=0.1):
    k=max(1,int(alpha*len(losses))); vals,_=torch.topk(losses,k,largest=True); return vals.mean()

# =========================
# ARTL (TTL + HOVR)
# =========================
class ARTLWrapper(nn.Module):
    def __init__(self, model, h=0.9, hovr_lambda=1e-4):
        super().__init__()
        self.model = model
        self.xi = nn.Parameter(torch.tensor(1.0))
        self.h = h
        self.hovr_lambda = hovr_lambda
    def forward(self,x): return self.model(x)

def ttl_with_augparam(losses, xi, h):
    tl = trimmed_mean(losses, h=h)
    return tl * torch.relu(xi)

def hovr_penalty_random(model, M=10):
    x_rand = torch.rand(M,1,IMG_SIZE,IMG_SIZE, device=device, requires_grad=True)
    y_rand = model(x_rand)
    g = torch.autograd.grad(y_rand, x_rand, torch.ones_like(y_rand), create_graph=True)[0]
    return (g.view(M,-1)**2).sum(1).mean()

# =========================
# AWP
# =========================
class AdvWeightPerturb:
    def __init__(self, model, gamma=1e-2, eps=1e-3):
        self.model=model; self.gamma=gamma; self.eps=eps; self.backup={}
    def perturb(self):
        self.backup={}
        for n,p in self.model.named_parameters():
            if p.requires_grad and p.grad is not None:
                self.backup[n]=p.data.clone()
                grad = p.grad
                norm = torch.norm(p.detach())
                if not torch.isfinite(norm) or norm==0:
                    continue
                r = self.gamma * grad / (norm + 1e-12)
                p.data.add_(r).clamp_(p.data-self.eps, p.data+self.eps)
    def restore(self):
        for n,p in self.model.named_parameters():
            if n in self.backup: p.data.copy_(self.backup[n])
        self.backup={}

def randomized_adversarial_loss(model, x, y, sigma=1e-3, use_second=True):
    """
    Randomized Adversarial Training (RAT, Jin et al. CVPR 2023)
    """
    pred_clean = model(x)
    loss0 = ((pred_clean - y)**2).mean()

    model_noisy = copy.deepcopy(model)
    with torch.no_grad():
        for p in model_noisy.parameters():
            if p.requires_grad:
                p.add_(sigma * torch.randn_like(p))

    pred_noisy = model_noisy(x)
    loss_noisy = ((pred_noisy - y)**2).mean()

    loss1 = loss_noisy - loss0
    loss2 = (loss1**2) if use_second else 0.0

    loss = loss0 + 0.5 * loss1 + 0.1 * loss2
    return loss

# =========================
# Train / Eval
# =========================
def train_one_epoch(model, opt, loader, mode, artl=None):
    model.train(); total=0.0; n=0
    awp = AdvWeightPerturb(model) if mode=="awp" else None

    for x,y in loader:
        x,y = x.to(device), y.to(device)
        opt.zero_grad()

        # --- Baselines ---
        if mode=="standard":
            loss = mse_per_sample(model(x), y).mean()
        elif mode=="robust_huber":
            loss = huber_per_sample(model(x),y,1.0).mean()
        elif mode=="robust_tukey":
            loss = tukey_per_sample(model(x),y,4.685).mean()
        elif mode=="trim":
            loss = trimmed_mean(mse_per_sample(model(x),y),0.9)
        elif mode=="proposed":
            p = model(x); tl = ttl_with_augparam(mse_per_sample(p,y), artl.xi, artl.h)
            hovr = hovr_penalty_random(model, M=HOVR_M)
            loss = tl + artl.hovr_lambda * hovr
        elif mode=="dro":
            loss = cvar_mean(mse_per_sample(model(x),y), alpha=0.1)
        elif mode=="awp":
            p = model(x); base_loss = mse_per_sample(p,y).mean()
            base_loss.backward(); awp.perturb()
            p2 = model(x); robust_loss = mse_per_sample(p2,y).mean()
            robust_loss.backward(); awp.restore(); opt.step()
            total += float((base_loss+robust_loss).detach().cpu())*x.size(0); n+=x.size(0)
            continue
        elif mode == "rat":
            loss = randomized_adversarial_loss(model, x, y, sigma=1e-3, use_second=True)
        else:
            raise ValueError(mode)

        loss.backward(); opt.step()
        total += float(loss.detach().cpu()) * x.size(0); n += x.size(0)
    return total/max(1,n)

@torch.no_grad()
def evaluate(model, loader, trim_h=0.9):
    model.eval(); mse=0; trim=0; n=0
    for x,y in loader:
        x,y=x.to(device),y.to(device)
        p=model(x); per=mse_per_sample(p,y)
        mse += per.mean().item()*len(x)
        trim+= trimmed_mean(per,trim_h).item()*len(x)
        n += len(x)
    return dict(mse=mse/n, trimmed_mse=trim/n)

# =========================
# Data
# =========================
train_ds = RotAngleMNIST(train=True,  angle_max_deg=ANGLE_MAX, size=IMG_SIZE, seed=42, outlier_ratio=OUTLIER_RATIO_TRAIN)
test_ds  = RotAngleMNIST(train=False, angle_max_deg=ANGLE_MAX, size=IMG_SIZE, seed=42, outlier_ratio=0.0)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, pin_memory=torch.cuda.is_available())
print("train outlier ratio =", train_ds.outlier_ratio, " | test outlier ratio =", test_ds.outlier_ratio)

# =========================
# Main Loop
# =========================
summary=[]
for mode in modes:
    print(f"\n=== {mode} ===")
    mse_list=[]; trim_list=[]
    for trial in tqdm(range(N_TRIALS), leave=False):
        set_seed(1000+trial)
        base = SmallCNN().to(device)

        if mode=="proposed":
            artl = ARTLWrapper(base, h=0.9, hovr_lambda=1e-4).to(device)
            params = list(base.parameters()) + [artl.xi]
            opt = torch.optim.SGD(params, lr=LR, weight_decay=WEIGHT_DECAY)
            for ep in range(EPOCHS):
                train_one_epoch(base, opt, train_loader, mode, artl=artl)
            m = evaluate(base, test_loader)
            mse_list.append(m["mse"]); trim_list.append(m["trimmed_mse"])
            continue
        else:
            opt = torch.optim.Adam(base.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        for ep in range(EPOCHS):
            train_one_epoch(base, opt, train_loader, mode)

        m = evaluate(base, test_loader)
        mse_list.append(m["mse"]); trim_list.append(m["trimmed_mse"])

    row = dict(mode=mode,
               mse_mean=float(np.mean(mse_list)), mse_sd=float(np.std(mse_list)),
               trimmed_mse_mean=float(np.mean(trim_list)), trimmed_mse_sd=float(np.std(trim_list)))
    summary.append(row)
    print(f"{mode:12s}  MSE={row['mse_mean']:.4f}±{row['mse_sd']:.4f}  "
          f"TrimMSE={row['trimmed_mse_mean']:.4f}±{row['trimmed_mse_sd']:.4f}")

df = pd.DataFrame(summary)
out_csv = os.path.join(OUTDIR, "summary_mean_sd.csv")
df.to_csv(out_csv, index=False)
print("\n Summary saved to:", out_csv)

from IPython.display import display
display(df.round(5))


100%|██████████| 9.91M/9.91M [00:00<00:00, 20.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 479kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.46MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.0MB/s]


train outlier ratio = 0.05  | test outlier ratio = 0.0

=== proposed ===


  0%|          | 0/5 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:829: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:179.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


proposed      MSE=0.8287±0.0058  TrimMSE=0.6646±0.0010

=== standard ===


standard      MSE=1.6248±1.3558  TrimMSE=1.3002±1.1980

=== robust_huber ===


...

trim          MSE=6.6903±6.5041  TrimMSE=6.0300±6.0986

=== dro ===


dro           MSE=2.8549±1.5594  TrimMSE=2.3765±1.4144

=== awp ===


awp           MSE=1.6080±1.0406  TrimMSE=1.2719±0.9309

=== rat ===


rat           MSE=2.0668±1.6712  TrimMSE=1.6925±1.5009

 Summary saved to: ./results_adv5/summary_mean_sd.csv


,mode,mse_mean,mse_sd,trimmed_mse_mean,trimmed_mse_sd
0,proposed,0.82865,0.00577,0.66461,0.00104
1,standard,1.62485,1.35577,1.30022,1.19799
2,robust_huber,2.14747,1.38580,1.74101,1.27148
3,robust_tukey,1.68571,2.06416,1.36646,1.88647
4,trim,6.69027,6.50414,6.02999,6.09858
5,dro,2.85494,1.55941,2.37648,1.41436
6,awp,1.60797,1.04055,1.27188,0.93089
7,rat,2.06681,1.67120,1.69252,1.50087
